# U9-01 · Caso práctico (1): comprensión de los datos de telemarketing

*Especialista en Inteligencia Artificial (IFCD107) · Unidad 9*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador; la primera vez tarda unos segundos en cargar las bibliotecas.

> Datos: Moro, S., Rita, P. y Cortez, P. (2014). *Bank Marketing*. UCI, CC BY 4.0. https://doi.org/10.24432/C5K306 · Moro, S., Cortez, P. y Rita, P. (2014). *Decision Support Systems*, 62, 22–31. https://doi.org/10.1016/j.dss.2014.03.001

## 9.2 · Comprensión de los datos

`x1_cargar.py`

In [ ]:
from banco import d

print(d.shape, "| filas repetidas:", d.duplicated().sum())
num = d.select_dtypes("number").columns
print("columnas numéricas:", len(num), "| de texto:",
      d.shape[1] - len(num))
desc = [c for c in d.columns if (d[c] == "unknown").any()]
print("valores 'unknown':", {c: int((d[c] == "unknown").sum())
                             for c in desc})
print("pdays = 999 (sin contacto previo):",
      f"{(d.pdays == 999).mean():.1%}")
print("default = yes:", int((d.default == "yes").sum()))

`x8_validar.py`

In [ ]:
from banco import d

reglas = {
    "edad entre 17 y 100": d.age.between(17, 100).all(),
    "llamadas de la campaña >= 1": (d.campaign >= 1).all(),
    "pdays: 999 o entre 0 y 30": (d.pdays.eq(999)
                                  | d.pdays.between(0, 30)).all(),
    "previous = 0 si nunca contactado": (d.previous[d.poutcome ==
                                         "nonexistent"] == 0).all(),
    "poutcome conocido si pdays != 999": (d.poutcome[d.pdays != 999]
                                          != "nonexistent").all(),
    "y solo 0 o 1": d.y.isin([0, 1]).all()}
for r, ok in reglas.items():
    print(f"{'OK   ' if ok else 'FALLA'} {r}")
raro = d[(d.pdays == 999) & (d.poutcome != "nonexistent")]
print("sin contacto reciente pero con campaña previa:", len(raro),
      "| resultado:", raro.poutcome.value_counts().to_dict())

`x2_objetivo.py`

In [ ]:
from banco import d, hist, X_e, X_p, X_f

print(f"suscriben: {d.y.sum()} de {len(d)} ({d.y.mean():.1%})")
print(d.groupby("año").y.agg(["size", "mean"]).round(3).to_string())
print("meses de 2010:", d[d.año == 2010].month.unique().tolist())
print("desarrollo (2008-2009):", hist.sum(), "| entrenamiento",
      len(X_e), "| prueba", len(X_p), "| futuro (2010)", len(X_f))

`x3_duracion.py`

In [ ]:
from sklearn.metrics import roc_auc_score
from banco import d

print(d.groupby("y").duration.median().to_string())
print("llamadas de 0 segundos:", (d.duration == 0).sum(),
      "| suscriben:", d.y[d.duration == 0].sum())
print(f"AUC usando solo la duración: "
      f"{roc_auc_score(d.y, d.duration):.3f}")

`x4_categoricas.py`

In [ ]:
from banco import d, hist

h = d[hist]
for c in ["poutcome", "contact", "job", "month"]:
    t = h.groupby(c).y.agg(["size", "mean"]).sort_values("mean")
    t["mean"] = (100 * t["mean"]).round(1)
    if len(t) > 4:                # 2 peores y 2 mejores
        t = t.iloc[[0, 1, -2, -1]]
    t.columns = ["n", "% suscribe"]
    print(f"--- {c}"); print(t.T.to_string(header=True))

`x7_desconocidos.py`

In [ ]:
from banco import d, hist

h = d[hist]
for c in ["default", "housing", "loan"]:
    t = h.groupby(c).y.agg(["size", "mean"])
    t["mean"] = (100 * t["mean"]).round(1)
    t.columns = ["n", "% suscribe"]
    print(f"--- {c}"); print(t.T.to_string())
u = (h[["job", "marital", "education", "default", "housing",
        "loan"]] == "unknown").sum(axis=1)
print("clientes con algún 'unknown':", f"{(u > 0).mean():.1%}")

`x5_numericas.py`

In [ ]:
import pandas as pd
from banco import d, hist

h = d[hist]
print(h[["age", "campaign", "previous"]].describe()
      .loc[["mean", "50%", "max"]].round(1).to_string())
tramos = pd.cut(h.campaign, [0, 1, 2, 3, 5, 10, 60])
print("% suscribe según las llamadas de esta campaña:")
print((100 * h.groupby(tramos, observed=True).y.mean()).round(1)
      .to_string())
edad = pd.cut(h.age, [16, 25, 40, 60, 100])
print("% suscribe según la edad:")
print((100 * h.groupby(edad, observed=True).y.mean()).round(1)
      .to_string())

`x9_segmentos.py`

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import make_column_transformer
from banco import d, hist

h = d[hist]
cliente = ["age", "job", "marital", "education", "housing", "loan"]
prep = make_column_transformer(
    (StandardScaler(), ["age"]),
    (OneHotEncoder(sparse_output=False), cliente[1:]))
Z = prep.fit_transform(h[cliente])
g = KMeans(5, n_init=10, random_state=0).fit_predict(Z)
r = h.assign(grupo=g).groupby("grupo").agg(
    clientes=("y", "size"), edad=("age", "median"),
    jubilado=("job", lambda s: (s == "retired").mean()),
    obrero=("job", lambda s: (s == "blue-collar").mean()),
    univ=("education", lambda s: (s == "university.degree").mean()),
    casado=("marital", lambda s: (s == "married").mean()),
    suscribe=("y", "mean"))
cols = ["jubilado", "obrero", "univ", "casado", "suscribe"]
r[cols] = (100 * r[cols]).round(1)       # porcentajes
print(r.sort_values("suscribe").to_string())

`x6_contexto.py`

In [ ]:
import matplotlib.pyplot as plt
from banco import d

eco = ["emp.var.rate", "cons.price.idx", "cons.conf.idx",
       "euribor3m", "nr.employed"]
print(d.groupby("año")[eco].mean().round(2).T.to_string())
c = d[eco + ["y"]].corr().round(2)
print("correlación con y:", c.y.drop("y").to_dict())
print("euribor y empleados:", c.loc["euribor3m", "nr.employed"])
fig, e1 = plt.subplots(figsize=(4.6, 2.0), layout="constrained")
bloque = d.index // 1000                   # bloques de 1000 llamadas
g = d.groupby(bloque).agg(eur=("euribor3m", "mean"),
                          tasa=("y", "mean"))
e1.plot(g.index, g.eur, label="Euríbor a 3 meses (%)")
e1.set_xlabel("llamadas en orden cronológico (miles)")
e1.set_ylabel("Euríbor (%)")
e2 = e1.twinx()
e2.plot(g.index, 100 * g.tasa, ls="--", color="C1",
        label="% que suscribe")
e2.set_ylabel("% que suscribe")
e1.legend(loc="upper center", fontsize=6)
e2.legend(loc="center left", fontsize=6)
plt.show()

**Prueba.** Calcula la tasa de suscripción por día de la semana. ¿Hay diferencias relevantes?